In [1]:
import numpy as np
import time
from dataclasses import dataclass
from typing import Callable, Optional
import matplotlib.pyplot as plt

In [2]:
from simulator import generate_hierarchical
from posteriors import *
from samplers_hierarchical import *

In [3]:
rng = np.random.default_rng(221)

print("Generating data")
ds = generate_hierarchical(rng=rng, truth_model = "power_law")
K = len(ds.seasons)
print(f"K = {K}, "
     f"True means = {ds.phi_k}")


Generating data
K = 10, True means = [1.23580097e-05 9.27441331e-06 8.02758220e-06 8.77680051e-06
 1.34651853e-05 1.00865273e-05 1.08705475e-05 1.33043578e-05
 6.35077821e-06 1.12390328e-05]


In [4]:
param_names = []

for i in range(K):
    param_names.append(f"log_phi_{i+1}")

param_names.extend(["gamma", "log_eta", "delta", "mu_phi", "log_sigma_phi"])
param_names

['log_phi_1',
 'log_phi_2',
 'log_phi_3',
 'log_phi_4',
 'log_phi_5',
 'log_phi_6',
 'log_phi_7',
 'log_phi_8',
 'log_phi_9',
 'log_phi_10',
 'gamma',
 'log_eta',
 'delta',
 'mu_phi',
 'log_sigma_phi']

In [5]:
ds.true_params

{'mu_phi': np.float64(-11.512925464970229),
 'sigma_phi': 0.3,
 'gamma': 2.5,
 'eta': 1e-05,
 'delta': 3.7,
 'truth_model': 'power_law',
 'K': 10}

In [6]:
theta_true = np.log(ds.phi_k).tolist()
theta_true.extend([ds.true_params["gamma"], np.log(ds.true_params["eta"]),
                   ds.true_params["delta"], ds.true_params["mu_phi"], np.log(ds.true_params["sigma_phi"])])
theta_true = np.array(theta_true)
print(f"True params, (transformed) are {theta_true}")

true_values = {
    **{f"log_phi_{k+1}": np.log(ds.phi_k[k]) for k in range(K)},
    "gamma": ds.true_params["gamma"],
    "log_eta": np.log(ds.true_params["eta"]),
    "delta": ds.true_params["delta"],
    "mu_phi": ds.true_params["mu_phi"],
    "log_sigma_phi": np.log(ds.true_params["sigma_phi"]),
}


True params, (transformed) are [-11.30120614 -11.58825121 -11.73262717 -11.64339862 -11.21540307
 -11.50430996 -11.42945349 -11.22741892 -11.9669332  -11.39611777
   2.5        -11.51292546   3.7        -11.51292546  -1.2039728 ]


In [7]:
prior_type = "lognormal_gamma"
parameterization = "centered"

def log_post(theta):
    return log_posterior_hierarchical(theta, ds.seasons, parameterization, prior_type)

def grad_log_post(theta):
    return grad_log_posterior_hierarchical(theta, ds.seasons, parameterization, prior_type)

In [8]:
theta_init = theta_true + rng.normal(0, 0.1, size=K+5)

In [9]:
rwmh_results = run_multiple_chains(
        run_rwmh,
        theta_init=theta_init,
        n_chains=4,
        init_strategy="jitter",
        init_scale=0.001,
        rng=rng,
        log_posterior_fn=log_post,
        n_iterations=100000,
        n_burnin=20000,
        adapt_proposal=True,
        param_names=param_names,
    )

Iteration 5000/100000: accept rate = 0.236, scale = 0.902, elapsed = 1.4s
Iteration 10000/100000: accept rate = 0.243, scale = 0.902, elapsed = 2.9s
Iteration 15000/100000: accept rate = 0.244, scale = 0.902, elapsed = 4.3s
Iteration 20000/100000: accept rate = 0.231, scale = 0.902, elapsed = 5.7s
Iteration 25000/100000: accept rate = 0.232, scale = 0.902, elapsed = 7.2s
Iteration 30000/100000: accept rate = 0.227, scale = 0.902, elapsed = 8.6s
Iteration 35000/100000: accept rate = 0.225, scale = 0.902, elapsed = 10.0s
Iteration 40000/100000: accept rate = 0.217, scale = 0.902, elapsed = 11.6s
Iteration 45000/100000: accept rate = 0.223, scale = 0.902, elapsed = 13.3s
Iteration 50000/100000: accept rate = 0.225, scale = 0.902, elapsed = 14.7s
Iteration 55000/100000: accept rate = 0.221, scale = 0.902, elapsed = 16.3s
Iteration 60000/100000: accept rate = 0.221, scale = 0.902, elapsed = 17.7s
Iteration 65000/100000: accept rate = 0.221, scale = 0.902, elapsed = 19.4s
Iteration 70000/100

In [10]:
rwmh_cov = estimate_dense_precond_from_rwmh(rwmh_results, ridge=1e-6)

In [11]:
mala_results = run_multiple_chains(
        run_mala,
        theta_init=theta_init,
        n_chains=4,
        init_strategy="jitter",
        init_scale=0.001,
        rng=rng,
        log_posterior_fn=log_post,
        grad_log_posterior_fn=grad_log_post,
        n_iterations=100000,
        n_burnin=20000,
        step_size=1e-3,
        adapt_step=True,
        adapt_until=20000,
        target_accept=0.57,
        param_names=param_names,
        precond=rwmh_cov,
        adapt_precond=False,
        precond_type="dense",
        normalize_precond=True,
    )

Iteration 5000/100000: accept rate = 0.719, step_size = 0.00465276, elapsed = 2.4s
Iteration 10000/100000: accept rate = 0.647, step_size = 0.00724994, elapsed = 4.9s
Iteration 15000/100000: accept rate = 0.611, step_size = 0.00435836, elapsed = 7.5s
Iteration 20000/100000: accept rate = 0.603, step_size = 0.00736893, elapsed = 10.1s
Iteration 25000/100000: accept rate = 0.594, step_size = 0.00736893, elapsed = 12.6s
Iteration 30000/100000: accept rate = 0.503, step_size = 0.00736893, elapsed = 15.1s
Iteration 35000/100000: accept rate = 0.479, step_size = 0.00736893, elapsed = 17.8s
Iteration 40000/100000: accept rate = 0.488, step_size = 0.00736893, elapsed = 20.2s
Iteration 45000/100000: accept rate = 0.493, step_size = 0.00736893, elapsed = 22.8s
Iteration 50000/100000: accept rate = 0.506, step_size = 0.00736893, elapsed = 25.2s
Iteration 55000/100000: accept rate = 0.522, step_size = 0.00736893, elapsed = 27.6s
Iteration 60000/100000: accept rate = 0.530, step_size = 0.00736893, 

In [12]:
print_diagnostics_multi_hierarchical(
    {"RWMH": rwmh_results, "MALA": mala_results},
    parameterization="centered",
    K=10,
    latent_display="log_phi",
    true_values=true_values,
)

save_traceplots_multi_hierarchical(
    rwmh_results,
    "traceplots_hierarchical_rwmh_centered.png",
    parameterization="centered",
    K=10,
    latent_display="log_phi",
)

save_traceplots_multi_hierarchical(
    mala_results,
    "traceplots_hierarchical_mala_centered.png",
    parameterization="centered",
    K=10,
    latent_display="log_phi",
)


Sampler      Chains  Accept%  Time(s)ESS(log_phi_1)ESS(log_phi_2)ESS(log_phi_3)ESS(log_phi_4)ESS(log_phi_5)ESS(log_phi_6)ESS(log_phi_7)ESS(log_phi_8)ESS(log_phi_9)ESS(log_phi_10)ESS(gamma)ESS(log_eta)ESS(delta)ESS(mu_phi)ESS(log_sigma_phi)
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
RWMH              4    0.216    121.2        254        271        352        367        617        257        319        293        309        296        442        265        460        293        266
MALA              4    0.495    178.1        547        444        738        690        667        446        464        506        369        506        494        393        663        474        390

Sampler      Chains  Accept%  Time(s)Rhat(log_phi_1)Rhat(log_phi_2)Rhat(log_phi_3)Rhat(log_phi_4)Rhat(log_phi_5)Rhat(log_phi_6)Rhat(log